# The Attention Mechanism for Protein Sequences

This notebook explores the attention mechanism and its application to protein sequence modeling.

**Learning Objectives:**
- Implement scaled dot-product attention from scratch
- Understand multi-head attention
- Build a simple transformer encoder
- Visualize attention patterns in protein sequences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install torch numpy matplotlib seaborn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Understanding Attention

The attention mechanism computes weighted combinations of values based on query-key similarities:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        query: (batch, seq_len_q, d_k) or (batch, heads, seq_len_q, d_k)
        key: (batch, seq_len_k, d_k) or (batch, heads, seq_len_k, d_k)
        value: (batch, seq_len_k, d_v) or (batch, heads, seq_len_k, d_v)
        mask: optional mask for padding or causal attention
    
    Returns:
        output: weighted combination of values
        attention_weights: attention probabilities
    """
    d_k = query.size(-1)
    
    # Step 1: Compute attention scores (dot product of Q and K)
    # (batch, ..., seq_len_q, d_k) @ (batch, ..., d_k, seq_len_k)
    # -> (batch, ..., seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(-2, -1))
    
    # Step 2: Scale by sqrt(d_k)
    scores = scores / math.sqrt(d_k)
    
    # Step 3: Apply mask (optional)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 4: Softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    # Handle NaN from all-masked rows
    attention_weights = attention_weights.nan_to_num(0)
    
    # Step 5: Apply attention to values
    output = torch.matmul(attention_weights, value)
    
    return output, attention_weights

In [ ]:
# Simple example: 4-token sequence
batch_size = 1
seq_len = 4
d_k = 8

# Random Q, K, V
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_k)

output, attention = scaled_dot_product_attention(Q, K, V)

print(f"Query shape: {Q.shape}")
print(f"Key shape: {K.shape}")
print(f"Value shape: {V.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention shape: {attention.shape}")
print(f"\nAttention weights (should sum to 1 per row):")
print(attention[0])
print(f"\nRow sums: {attention[0].sum(dim=-1)}")

In [ ]:
# Visualize attention weights
plt.figure(figsize=(6, 5))
sns.heatmap(attention[0].numpy(), annot=True, fmt='.2f', cmap='Blues',
            xticklabels=['Pos 0', 'Pos 1', 'Pos 2', 'Pos 3'],
            yticklabels=['Pos 0', 'Pos 1', 'Pos 2', 'Pos 3'])
plt.xlabel('Key Position (attending to)')
plt.ylabel('Query Position (from)')
plt.title('Attention Weights')
plt.show()

## Part 2: Attention with Masks

Masks are used to:
1. **Padding mask**: Ignore padding tokens
2. **Causal mask**: Prevent looking at future tokens (for autoregressive models)

In [ ]:
# Padding mask example
# Suppose position 3 is padding
seq_len = 4
padding_mask = torch.ones(1, 1, seq_len)  # (batch, 1, seq_len) for broadcasting
padding_mask[0, 0, 3] = 0  # Position 3 is padding

output_masked, attention_masked = scaled_dot_product_attention(Q, K, V, mask=padding_mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].set_title('Without Padding Mask')
sns.heatmap(attention[0].numpy(), annot=True, fmt='.2f', cmap='Blues', ax=axes[0])

axes[1].set_title('With Padding Mask (Pos 3 = padding)')
sns.heatmap(attention_masked[0].numpy(), annot=True, fmt='.2f', cmap='Blues', ax=axes[1])

plt.tight_layout()
plt.show()

print("Notice: No attention goes to position 3 (the padding token)")

In [ ]:
# Causal mask example (for autoregressive models)
def create_causal_mask(seq_len):
    """Create lower triangular mask to prevent attending to future."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

causal_mask = create_causal_mask(4)
print("Causal mask:")
print(causal_mask[0, 0])

output_causal, attention_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

plt.figure(figsize=(6, 5))
sns.heatmap(attention_causal[0].numpy(), annot=True, fmt='.2f', cmap='Blues')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Causal Attention (can only attend to past)')
plt.show()

print("\nNotice: Each position can only attend to itself and previous positions")

## Part 3: Multi-Head Attention

Multi-head attention runs multiple attention operations in parallel, allowing the model to attend to different aspects of the input:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention from 'Attention Is All You Need'.
    """
    
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        # Linear projections for Q, K, V
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        
        # Output projection
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None, return_attention=False):
        """
        Args:
            query: (batch, seq_len_q, embed_dim)
            key: (batch, seq_len_k, embed_dim)
            value: (batch, seq_len_k, embed_dim)
            mask: optional attention mask
        
        Returns:
            output: (batch, seq_len_q, embed_dim)
        """
        batch_size, seq_len_q, _ = query.shape
        seq_len_k = key.shape[1]
        
        # Project Q, K, V
        Q = self.q_proj(query)  # (batch, seq_len_q, embed_dim)
        K = self.k_proj(key)
        V = self.v_proj(value)
        
        # Reshape for multi-head: (batch, seq_len, num_heads, head_dim) -> (batch, num_heads, seq_len, head_dim)
        Q = Q.view(batch_size, seq_len_q, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, seq_len_k, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, seq_len_k, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Compute attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Apply dropout to attention weights
        attn_weights = self.dropout(attn_weights)
        
        # Reshape back: (batch, num_heads, seq_len, head_dim) -> (batch, seq_len, embed_dim)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.embed_dim)
        
        # Final projection
        output = self.out_proj(attn_output)
        
        if return_attention:
            return output, attn_weights
        return output

In [ ]:
# Test multi-head attention
embed_dim = 64
num_heads = 4
seq_len = 8
batch_size = 2

mha = MultiHeadAttention(embed_dim, num_heads)

x = torch.randn(batch_size, seq_len, embed_dim)
output, attention = mha(x, x, x, return_attention=True)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention shape: {attention.shape}")
print(f"  -> (batch={batch_size}, heads={num_heads}, query_len={seq_len}, key_len={seq_len})")

In [ ]:
# Visualize attention from different heads
fig, axes = plt.subplots(1, num_heads, figsize=(16, 3.5))

for head in range(num_heads):
    ax = axes[head]
    sns.heatmap(attention[0, head].detach().numpy(), cmap='Blues', ax=ax,
                cbar=head == num_heads - 1)
    ax.set_title(f'Head {head + 1}')
    ax.set_xlabel('Key' if head == num_heads // 2 else '')
    ax.set_ylabel('Query' if head == 0 else '')

plt.suptitle('Attention Patterns Across Heads', y=1.05)
plt.tight_layout()
plt.show()

print("Different heads learn different attention patterns!")

## Part 4: Transformer Block

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""
    
    def __init__(self, embed_dim, max_len=1000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        
        pe[:, 0::2] = torch.sin(position * div_term)  # Even indices
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd indices
        
        # Register as buffer (not a parameter)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, embed_dim)
    
    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
# Visualize positional encoding
pe = PositionalEncoding(embed_dim=64, max_len=100)
pe_matrix = pe.pe[0, :50].numpy()

plt.figure(figsize=(12, 4))
plt.imshow(pe_matrix.T, aspect='auto', cmap='RdBu')
plt.colorbar(label='Value')
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Sinusoidal Positional Encoding')
plt.show()

In [ ]:
class TransformerBlock(nn.Module):
    """Single transformer encoder block."""
    
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        
        # Multi-head self-attention
        self.attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        
        # Feed-forward network
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(embed_dim)
    
    def forward(self, x, mask=None, return_attention=False):
        # Self-attention with residual (pre-norm)
        normed = self.norm1(x)
        if return_attention:
            attn_out, attn_weights = self.attention(normed, normed, normed, mask, return_attention=True)
        else:
            attn_out = self.attention(normed, normed, normed, mask)
        x = x + attn_out
        
        # Feed-forward with residual (pre-norm)
        x = x + self.ff(self.norm2(x))
        
        if return_attention:
            return x, attn_weights
        return x

In [ ]:
class TransformerEncoder(nn.Module):
    """Transformer encoder for protein sequences."""
    
    def __init__(self, vocab_size=21, embed_dim=128, num_heads=4, 
                 num_layers=4, ff_dim=512, max_len=512, dropout=0.1):
        super().__init__()
        
        # Token embedding
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Positional encoding
        self.pos_encoding = PositionalEncoding(embed_dim, max_len, dropout)
        
        # Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x, mask=None, return_attention=False):
        # x: (batch, seq_len) token indices
        
        # Embed and add positional encoding
        x = self.embedding(x)
        x = self.pos_encoding(x)
        
        # Apply transformer blocks
        attentions = []
        for layer in self.layers:
            if return_attention:
                x, attn = layer(x, mask, return_attention=True)
                attentions.append(attn)
            else:
                x = layer(x, mask)
        
        x = self.norm(x)
        
        if return_attention:
            return x, attentions
        return x

In [ ]:
# Create a transformer encoder
model = TransformerEncoder(
    vocab_size=21,  # 20 amino acids + padding
    embed_dim=64,
    num_heads=4,
    num_layers=4,
    ff_dim=256
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"\nModel architecture:")
print(model)

## Part 5: Applying to Protein Sequences

In [ ]:
# Encode protein sequence
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i + 1 for i, aa in enumerate(AMINO_ACIDS)}  # 0 = padding

def encode_sequence(sequence):
    """Convert amino acid sequence to token indices."""
    return torch.tensor([AA_TO_IDX.get(aa, 0) for aa in sequence])

# Example: Ubiquitin
ubiquitin = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
tokens = encode_sequence(ubiquitin).unsqueeze(0)  # Add batch dimension

print(f"Sequence: {ubiquitin}")
print(f"Length: {len(ubiquitin)}")
print(f"Token shape: {tokens.shape}")

In [ ]:
# Get embeddings and attention patterns
model.eval()
with torch.no_grad():
    embeddings, attentions = model(tokens, return_attention=True)

print(f"Embedding shape: {embeddings.shape}")
print(f"Number of attention layers: {len(attentions)}")
print(f"Attention shape per layer: {attentions[0].shape}")

In [ ]:
# Visualize attention patterns across layers and heads
fig, axes = plt.subplots(len(attentions), 4, figsize=(16, 4 * len(attentions)))

for layer_idx, attn in enumerate(attentions):
    for head_idx in range(4):
        ax = axes[layer_idx, head_idx]
        attn_matrix = attn[0, head_idx].numpy()
        
        # Show only first 30 positions for clarity
        im = ax.imshow(attn_matrix[:30, :30], cmap='Blues', aspect='auto')
        ax.set_title(f'Layer {layer_idx + 1}, Head {head_idx + 1}')
        
        if head_idx == 0:
            ax.set_ylabel('Query Position')
        if layer_idx == len(attentions) - 1:
            ax.set_xlabel('Key Position')

plt.suptitle('Attention Patterns Across Layers and Heads (First 30 positions)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Average attention across heads for each layer
fig, axes = plt.subplots(1, len(attentions), figsize=(5 * len(attentions), 4))

for layer_idx, attn in enumerate(attentions):
    ax = axes[layer_idx]
    avg_attn = attn[0].mean(dim=0).numpy()  # Average over heads
    
    im = ax.imshow(avg_attn[:40, :40], cmap='Blues', aspect='auto')
    ax.set_title(f'Layer {layer_idx + 1} (avg over heads)')
    ax.set_xlabel('Key Position')
    if layer_idx == 0:
        ax.set_ylabel('Query Position')

plt.colorbar(im, ax=axes, shrink=0.6)
plt.suptitle('Average Attention by Layer', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze attention to specific positions
# Let's look at which positions attend most to each other

# Average attention across all layers and heads
all_attn = torch.stack([a[0] for a in attentions]).mean(dim=(0, 1)).numpy()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full attention map
im = axes[0].imshow(all_attn, cmap='Blues', aspect='auto')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')
axes[0].set_title('Average Attention (All Layers & Heads)')
plt.colorbar(im, ax=axes[0])

# Attention strength by sequence separation
L = all_attn.shape[0]
separations = np.arange(L)
avg_by_sep = []

for sep in separations:
    if sep < L:
        diag_values = np.diagonal(all_attn, offset=sep)
        if len(diag_values) > 0:
            avg_by_sep.append(diag_values.mean())
        else:
            avg_by_sep.append(0)

axes[1].plot(separations[:50], avg_by_sep[:50], 'b-', linewidth=2)
axes[1].set_xlabel('Sequence Separation |i - j|')
axes[1].set_ylabel('Average Attention')
axes[1].set_title('Attention vs Sequence Separation')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The attention mechanism learns both local and long-range relationships!")

## Part 6: Protein Classification with Transformer

In [ ]:
class ProteinClassifier(nn.Module):
    """Transformer-based protein classifier."""
    
    def __init__(self, vocab_size=21, embed_dim=64, num_heads=4,
                 num_layers=4, ff_dim=256, num_classes=2, dropout=0.1):
        super().__init__()
        
        # Transformer encoder
        self.encoder = TransformerEncoder(
            vocab_size, embed_dim, num_heads, num_layers, ff_dim, dropout=dropout
        )
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes)
        )
    
    def forward(self, x, mask=None):
        # Encode
        hidden = self.encoder(x, mask)  # (batch, seq_len, embed_dim)
        
        # Global average pooling (with mask)
        if mask is not None:
            # mask: (batch, seq_len), 1 for valid, 0 for padding
            mask_expanded = mask.unsqueeze(-1)  # (batch, seq_len, 1)
            pooled = (hidden * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1)
        else:
            pooled = hidden.mean(dim=1)
        
        # Classify
        logits = self.classifier(pooled)
        
        return logits

In [ ]:
# Create classifier and test
classifier = ProteinClassifier(
    vocab_size=21,
    embed_dim=64,
    num_heads=4,
    num_layers=4,
    num_classes=2
)

# Test with random batch
batch_size = 8
seq_len = 50
x = torch.randint(1, 21, (batch_size, seq_len))  # Random sequences
mask = torch.ones(batch_size, seq_len)  # All valid

logits = classifier(x, mask)
print(f"Input shape: {x.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"\nPredicted probabilities:")
print(F.softmax(logits[:4], dim=-1))

## Summary

In this notebook, we learned:

1. **Scaled dot-product attention** computes weighted sums based on query-key similarity
2. **Multi-head attention** runs multiple attention operations in parallel
3. **Positional encoding** adds position information since transformers lack inherent ordering
4. **Transformer blocks** combine attention, feed-forward networks, and residual connections
5. **Attention patterns** reveal how the model relates different sequence positions

**Key insights for proteins:**
- Transformers can capture both local motifs and long-range interactions
- Different attention heads learn different relationship types
- Deeper layers often capture more abstract patterns
- Pre-trained protein transformers (like ESM) learn biologically meaningful attention